# 01 - Partisi Data untuk Federated Learning (TANPA SFM)

**Tujuan belajar:** langkah PERTAMA membangun percobaan FL — mengubah satu dataset
terpusat menjadi beberapa 'klien' yang masing-masing punya data sendiri.

**Prinsip:**
- **Tanpa SFM.** Kita pakai satu dataset (CIC-IDS2017) apa adanya; fitur numerik
  CICFlowMeter langsung, tidak ada pemetaan semantik.
- **IID dulu** (independent & identically distributed): tiap klien dapat potongan acak
  yang distribusinya mirip. Ini kondisi FL paling mudah dipahami. Non-IID menyusul di nb04.
- Label biner: benign=0, attack=1.

**Output:** file partisi per-klien `client_{i}_train.npz` / `client_{i}_test.npz` +
`global_test.npz` (test bersama untuk menilai model global) di folder `fl_data/`.

Dijalankan di AWS SageMaker. Dataset di-sync dari S3 bila belum ada lokal.

In [ ]:
import sys
!{sys.executable} -m pip install -q numpy pandas scikit-learn
print('deps OK')

In [ ]:
# === KONFIG ===
import os, glob, zipfile, subprocess, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

S3_BUCKET = 'ssh-detection-features-232032302717'
S3_RAW    = 's3://%s/datasets/CICD2017/' % S3_BUCKET   # satu dataset utk belajar
DATA_DIR  = './data_src'      # CSV mentah dari S3
OUT_DIR   = './fl_data'       # hasil partisi per-klien
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

K_CLIENTS = 4            # jumlah klien federasi
MAX_ROWS  = 200000       # batas baris (kendali memori; stratified)
TEST_FRAC = 0.2          # porsi test
SEED      = 42
np.random.seed(SEED)
FORCE_SYNC = False
print('K klien =', K_CLIENTS, '| dataset = CIC-IDS2017 (tanpa SFM)')

In [ ]:
# === Ambil dataset dari S3 bila folder lokal kosong (+ auto-unzip) ===
def has_csv(d): return len(glob.glob(os.path.join(d,'**','*.csv'),recursive=True))>0
def has_zip(d): return len(glob.glob(os.path.join(d,'**','*.zip'),recursive=True))>0
if FORCE_SYNC or not has_csv(DATA_DIR):
    print('sync dari', S3_RAW)
    subprocess.run(['aws','s3','sync',S3_RAW,DATA_DIR,'--only-show-errors'], check=True)
    for zp in glob.glob(os.path.join(DATA_DIR,'**','*.zip'),recursive=True):
        print('unzip', os.path.basename(zp))
        with zipfile.ZipFile(zp) as z: z.extractall(DATA_DIR)
else:
    print('CSV sudah ada di', DATA_DIR, '-> lewati download')
print('jumlah CSV:', len(glob.glob(os.path.join(DATA_DIR,'**','*.csv'),recursive=True)))

In [ ]:
# === Muat + bersihkan dataset (tanpa SFM, fitur numerik apa adanya) ===
def norm(c):
    return ' '.join(str(c).replace('\ufeff','').split()).lower()
META = ['flow id','flowid','source ip','src ip','destination ip','dst ip','source port',
        'src port','destination port','dst port','protocol','timestamp','unnamed']
frames=[]
for fp in sorted(glob.glob(os.path.join(DATA_DIR,'**','*.csv'),recursive=True)):
    df = pd.read_csv(fp, low_memory=False)
    df.columns = [norm(c) for c in df.columns]
    lab = [c for c in df.columns if 'label' in c]
    if not lab: continue
    df = df.rename(columns={lab[0]:'label'})
    frames.append(df)
raw = pd.concat(frames, ignore_index=True)
print('baris mentah gabungan:', len(raw))

# label biner
y = raw['label'].astype(str).str.lower().apply(lambda s: 0 if ('benign' in s or s=='normal') else 1).values
# fitur: buang metadata + label, numerik saja, bersihkan inf/nan
X = raw.drop(columns=['label'])
X = X[[c for c in X.columns if not any(m in c for m in META)]]
X = X.apply(pd.to_numeric, errors='coerce').replace([np.inf,-np.inf], np.nan)
X = X.dropna(axis=1, how='all')
X = X.fillna(X.median(numeric_only=True))
FEATURES = list(X.columns)
X = X.values.astype('float32')
print('fitur dipakai:', X.shape[1], '| attack%:', round(100*y.mean(),2))

In [ ]:
# === Stratified subsample (kendali memori) ===
from sklearn.model_selection import train_test_split
if len(X) > MAX_ROWS:
    idx,_ = train_test_split(np.arange(len(X)), train_size=MAX_ROWS, stratify=y, random_state=SEED)
    X, y = X[idx], y[idx]
print('setelah subsample:', X.shape, '| attack%:', round(100*y.mean(),2))

In [ ]:
# === Bagi GLOBAL train/test, lalu partisi train ke K klien IID ===
# global_test = test bersama untuk menilai MODEL GLOBAL hasil federasi.
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_FRAC, stratify=y, random_state=SEED)
np.savez(os.path.join(OUT_DIR,'global_test.npz'), X=Xte, y=yte)

# IID: acak lalu bagi rata ke K klien (stratified agar tiap klien mirip distribusinya)
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=K_CLIENTS, shuffle=True, random_state=SEED)
for i,(_, cidx) in enumerate(skf.split(Xtr, ytr)):
    Xc, yc = Xtr[cidx], ytr[cidx]
    # tiap klien punya train/test lokal sendiri juga
    Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=TEST_FRAC, stratify=yc, random_state=SEED)
    np.savez(os.path.join(OUT_DIR,f'client_{i}_train.npz'), X=Xc_tr, y=yc_tr)
    np.savez(os.path.join(OUT_DIR,f'client_{i}_test.npz'),  X=Xc_te, y=yc_te)
    print(f'klien {i}: train {len(yc_tr)} (attack {round(100*yc_tr.mean(),1)}%), test {len(yc_te)}')
print('\nglobal_test:', len(yte), 'baris (attack', round(100*yte.mean(),1),'%)')

In [ ]:
# === Ringkasan partisi (untuk dipahami) ===
import json
summary = {'dataset':'CIC-IDS2017 (tanpa SFM)','n_features':len(FEATURES),
           'k_clients':K_CLIENTS,'partition':'IID (stratified)',
           'global_test_rows':int(len(yte)),'features':FEATURES}
json.dump(summary, open(os.path.join(OUT_DIR,'partition_summary.json'),'w'), indent=2)
print('Partisi selesai. File di', OUT_DIR)
print('Langkah berikut: 02_local_baseline (latih lokal & centralized sbg pembanding).')
import os as _os
for f in sorted(_os.listdir(OUT_DIR)): print('  ', f)